# 6.22 - CertCF Adult Minkowski Capsule Diagnostics

This notebook tests a first direct prototype of **full-dimensional certified merging** on the real Adult dataset.

For a local group of anchors with centers `a_1, ..., a_K`, we define a merged set as:

\[
\mathcal{M} = \mathrm{Conv}(a_1, \dots, a_K) \oplus \mathcal{B}_1(arepsilon)
\]

This is a **Minkowski capsule**:
- the simplex / convex hull of selected centers gives a tight certified skeleton
- the added `L1` ball gives full-dimensional volume

The certification route here is a practical prototype:
- compute the first hidden-layer bounds **exactly** over the simplex + `L1` ball
- propagate those bounds through the remaining ReLU MLP with interval bound propagation
- certify the merged capsule if the final binary logit-margin lower bound stays positive

This notebook is primarily about:
- whether Adult admits nontrivial certified capsule radii
- whether those capsules allow atlas compression


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from torch.utils.data import TensorDataset

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from certcf import CertCFAtlas, NearestOppositeClassClearanceStrategy
from dataset_specs import get_tabular_dataset_spec
from models.classifiers import TabularClassifier
from training.datamodules.adult import AdultDataModule
from training.lit_classifier import LitClassifier

torch.set_grad_enabled(False)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.titlesize': 12,
})

print({'device': str(DEVICE), 'seed': SEED})


In [ ]:
DATA_CFG = {
    'filepath': str(ROOT / 'data' / 'Adult' / 'raw.parquet'),
    'batch_size': 256,
    'val_fraction': 0.1,
    'test_fraction': 0.1,
    'seed': SEED,
    'num_workers': 0,
    'pca_enabled': False,
}

dm = AdultDataModule(**DATA_CFG)
dm.setup()
X_TRAIN, Y_TRAIN_TRUE = [t.numpy() for t in dm.train_ds.tensors]
X_TEST, Y_TEST_TRUE = [t.numpy() for t in dm.test_ds.tensors]

print({
    'train_shape': tuple(X_TRAIN.shape),
    'test_shape': tuple(X_TEST.shape),
    'train_class_counts': {int(c): int((Y_TRAIN_TRUE == c).sum()) for c in np.unique(Y_TRAIN_TRUE)},
    'test_class_counts': {int(c): int((Y_TEST_TRUE == c).sum()) for c in np.unique(Y_TEST_TRUE)},
})


In [ ]:
CKPT_PATH = ROOT / 'checkpoints' / 'adult_classifier' / 'best.ckpt'
SPEC = get_tabular_dataset_spec('adult')


def infer_tabular_classifier_dims_from_checkpoint(checkpoint: str | Path) -> tuple[list[int], int]:
    ckpt = torch.load(str(checkpoint), map_location='cpu', weights_only=False)
    state_dict = ckpt.get('state_dict', {})
    hidden_1 = state_dict.get('model.net.4.weight')
    output = state_dict.get('model.net.6.weight')
    if hidden_1 is None or output is None:
        raise KeyError('Could not infer hidden_dims / num_classes from the Adult checkpoint.')
    hidden_dims = [int(hidden_1.shape[1]), int(hidden_1.shape[0])]
    num_classes = int(output.shape[0])
    return hidden_dims, num_classes


def load_adult_model(checkpoint: str | Path, device: torch.device = DEVICE) -> torch.nn.Module:
    hidden_dims, num_classes = infer_tabular_classifier_dims_from_checkpoint(checkpoint)
    backbone = TabularClassifier(
        input_types=list(SPEC.input_types),
        cardinalities=list(SPEC.cardinalities),
        hidden_dims=hidden_dims,
        num_classes=num_classes,
        dropout=0.2,
    )
    lit = LitClassifier.load_from_checkpoint(str(checkpoint), model=backbone, map_location=str(device))
    net = lit.model.eval().to(device)
    net_no_dropout = torch.nn.Sequential(*[m for m in net.net if not isinstance(m, torch.nn.Dropout)])
    return net_no_dropout.eval().to(device)


@torch.no_grad()
def predict_np(model: torch.nn.Module, x: np.ndarray, device: torch.device = DEVICE) -> tuple[np.ndarray, np.ndarray]:
    x_t = torch.from_numpy(np.asarray(x, dtype=np.float32)).to(device)
    logits = model(x_t)
    probs = torch.softmax(logits, dim=1).cpu().numpy().astype(np.float32)
    preds = probs.argmax(axis=1)
    return preds, probs


MODEL = load_adult_model(CKPT_PATH, device=DEVICE)
Y_TRAIN_PRED, _ = predict_np(MODEL, X_TRAIN)
Y_TEST_PRED, _ = predict_np(MODEL, X_TEST)

print({
    'checkpoint': str(CKPT_PATH),
    'predicted_train_class_counts': {int(c): int((Y_TRAIN_PRED == c).sum()) for c in np.unique(Y_TRAIN_PRED)},
    'predicted_test_class_counts': {int(c): int((Y_TEST_PRED == c).sum()) for c in np.unique(Y_TEST_PRED)},
})


## Atlas and capsule setup

We build one normal Adult atlas on a prediction-aligned support subset, then try iterative local merging with Minkowski capsules.


In [ ]:
RUN_CFG = {
    'alpha': 0.45,
    'support_max_per_class': 600,
    'batch_size': 256,
    'norm': 1,
    'distance_norm': 1,
    'solver_maxiter': 500,
    'query_method': 'nearest_anchor',
    'proposal_top_center_neighbors': 20,
    'proposal_neighbor_pool': 12,
    'proposal_max_triples_per_anchor': 10,
    'proposal_max_quads_per_anchor': 8,
    'iter_max_passes': 12,
    'iter_top_pairs_per_pass': 800,
    'iter_top_triples_per_pass': 400,
    'iter_top_quads_per_pass': 250,
    'capsule_binary_search_steps': 10,
    'capsule_eps_floor': 1e-3,
}


def stratified_subsample(x: np.ndarray, y: np.ndarray, max_per_class: int | None, seed: int = SEED):
    if max_per_class is None:
        keep = np.arange(len(x))
        return x, y, keep
    rng = np.random.default_rng(seed)
    keep = []
    for cls in sorted(np.unique(y)):
        idx = np.where(y == cls)[0]
        if len(idx) > max_per_class:
            idx = rng.choice(idx, size=max_per_class, replace=False)
        keep.append(np.sort(idx))
    keep = np.sort(np.concatenate(keep))
    return x[keep], y[keep], keep


X_SUPPORT, Y_SUPPORT, SUPPORT_IDX = stratified_subsample(
    X_TRAIN,
    Y_TRAIN_PRED,
    max_per_class=RUN_CFG['support_max_per_class'],
)

print({
    'support_shape': tuple(X_SUPPORT.shape),
    'support_class_counts': {int(c): int((Y_SUPPORT == c).sum()) for c in np.unique(Y_SUPPORT)},
})

DS = TensorDataset(torch.from_numpy(X_SUPPORT).float(), torch.from_numpy(Y_SUPPORT).long())
ATLAS = CertCFAtlas(
    MODEL,
    DS,
    DEVICE,
    cnn=False,
    norm=RUN_CFG['norm'],
    distance_norm=RUN_CFG['distance_norm'],
    lirpa_method='backward',
    eps_strategy=NearestOppositeClassClearanceStrategy(alpha=RUN_CFG['alpha']),
    batch_size=RUN_CFG['batch_size'],
    default_query_method=RUN_CFG['query_method'],
    solver_maxiter=RUN_CFG['solver_maxiter'],
)

t0 = time.perf_counter()
ATLAS.build(build_unions=False, verbose=True)
ATLAS_BUILD_SECONDS = time.perf_counter() - t0
print({'atlas_build_seconds': float(ATLAS_BUILD_SECONDS)})


In [ ]:
def get_tabular_linear_layers(model: torch.nn.Module):
    base = model.net if hasattr(model, 'net') else model
    layers = list(base)
    if not (len(layers) == 5 and isinstance(layers[0], torch.nn.Linear) and isinstance(layers[1], torch.nn.ReLU)
            and isinstance(layers[2], torch.nn.Linear) and isinstance(layers[3], torch.nn.ReLU)
            and isinstance(layers[4], torch.nn.Linear)):
        raise TypeError('Expected dropout-stripped TabularClassifier net = Linear/ReLU/Linear/ReLU/Linear.')
    return layers[0], layers[2], layers[4]


L1, L2, L3 = get_tabular_linear_layers(MODEL)
W1 = L1.weight.detach().cpu().numpy().astype(np.float64)
b1 = L1.bias.detach().cpu().numpy().astype(np.float64)
W2 = L2.weight.detach().cpu().numpy().astype(np.float64)
b2 = L2.bias.detach().cpu().numpy().astype(np.float64)
W3 = L3.weight.detach().cpu().numpy().astype(np.float64)
b3 = L3.bias.detach().cpu().numpy().astype(np.float64)


def linear_interval(W: np.ndarray, b: np.ndarray, lower: np.ndarray, upper: np.ndarray):
    W_pos = np.maximum(W, 0.0)
    W_neg = np.minimum(W, 0.0)
    lower_out = W_pos @ lower + W_neg @ upper + b
    upper_out = W_pos @ upper + W_neg @ lower + b
    return lower_out, upper_out


def relu_interval(lower: np.ndarray, upper: np.ndarray):
    return np.maximum(lower, 0.0), np.maximum(upper, 0.0)


def certify_minkowski_capsule_binary(target_label: int, centers: np.ndarray, eps: float):
    centers = np.asarray(centers, dtype=np.float64)
    if centers.ndim != 2:
        raise ValueError('centers must have shape (K, d)')

    anchor_affines = centers @ W1.T
    dual_radius = np.max(np.abs(W1), axis=1) * float(eps)
    h1_lower = anchor_affines.min(axis=0) - dual_radius + b1
    h1_upper = anchor_affines.max(axis=0) + dual_radius + b1

    h1_lower, h1_upper = relu_interval(h1_lower, h1_upper)
    h2_lower, h2_upper = linear_interval(W2, b2, h1_lower, h1_upper)
    h2_lower, h2_upper = relu_interval(h2_lower, h2_upper)
    logits_lower, logits_upper = linear_interval(W3, b3, h2_lower, h2_upper)

    other_label = 1 - int(target_label)
    margin_lower = float(logits_lower[int(target_label)] - logits_upper[int(other_label)])
    return {
        'certified': bool(margin_lower > 0.0),
        'margin_lower': margin_lower,
    }


def binary_search_capsule_eps(target_label: int, centers: np.ndarray, eps_hi: float, steps: int):
    lo = 0.0
    hi = max(float(eps_hi), 0.0)
    if hi <= 0.0:
        zero_cert = certify_minkowski_capsule_binary(target_label, centers, eps=0.0)
        return {
            'certified': bool(zero_cert['certified']),
            'eps': 0.0,
            'margin_lower': float(zero_cert['margin_lower']),
        }

    zero_cert = certify_minkowski_capsule_binary(target_label, centers, eps=0.0)
    if not zero_cert['certified']:
        return {
            'certified': False,
            'eps': 0.0,
            'margin_lower': float(zero_cert['margin_lower']),
        }

    best = {'certified': True, 'eps': 0.0, 'margin_lower': float(zero_cert['margin_lower'])}
    for _ in range(int(steps)):
        mid = 0.5 * (lo + hi)
        cert = certify_minkowski_capsule_binary(target_label, centers, eps=mid)
        if cert['certified']:
            lo = mid
            best = {'certified': True, 'eps': float(mid), 'margin_lower': float(cert['margin_lower'])}
        else:
            hi = mid
    return best


def make_initial_capsule_regions(atlas, target_label: int):
    bd = atlas.bounds[target_label]
    regions = []
    for idx in range(len(bd['X'])):
        center = np.asarray(bd['X'][idx], dtype=np.float32)
        base_eps = float(bd['eps'][idx])
        regions.append({
            'region_key': f'class{target_label}_poly_{idx}',
            'center': center,
            'centers': center.reshape(1, -1),
            'member_count': 1,
            'source_ids': [int(idx)],
            'source_eps_map': {int(idx): float(base_eps)},
            'contained_source_ids': [int(idx)],
            'uncontained_source_ids': [],
            'min_source_eps': float(base_eps),
            'eps_cap_upper': float(base_eps),
            'certified_eps': float(base_eps),
            'margin_lower': np.nan,
        })
    return regions


def build_local_capsule_candidates(regions, cfg):
    if len(regions) < 2:
        return pd.DataFrame(), {}

    centers = np.stack([np.asarray(region['center'], dtype=np.float32) for region in regions], axis=0)
    center_dmat = np.abs(centers[:, None, :] - centers[None, :, :]).sum(axis=2)
    candidate_map = {}

    def register_subset(indices, source_tag):
        indices = tuple(sorted({int(idx) for idx in indices}))
        if len(indices) < 2:
            return
        key = '|'.join(regions[idx]['region_key'] for idx in indices)
        if key in candidate_map:
            candidate_map[key]['proposal_sources'].add(source_tag)
            return
        subset = [regions[idx] for idx in indices]
        anchor_centers = np.concatenate([region['centers'] for region in subset], axis=0)
        unique_centers = np.unique(anchor_centers, axis=0)
        eps_hi = min(float(region['eps_cap_upper']) for region in subset)
        source_eps_map = {}
        for region in subset:
            source_eps_map.update(region['source_eps_map'])
        source_ids = sorted(source_eps_map)
        min_source_eps = min(source_eps_map.values()) if source_eps_map else np.nan
        candidate_map[key] = {
            'candidate_key': key,
            'region_indices': indices,
            'subset': subset,
            'subset_size': int(len(indices)),
            'member_count_before': int(sum(region['member_count'] for region in subset)),
            'center_dist': float(center_dmat[np.ix_(indices, indices)].max()),
            'centers': unique_centers.astype(np.float32),
            'n_anchor_centers': int(unique_centers.shape[0]),
            'eps_cap_upper': float(eps_hi),
            'source_ids': source_ids,
            'source_eps_map': source_eps_map,
            'min_source_eps': float(min_source_eps),
            'proposal_sources': {source_tag},
        }

    for i in range(len(regions)):
        order = np.argsort(center_dmat[i])
        neighbors = [int(j) for j in order[1:1 + int(cfg['proposal_top_center_neighbors'])]]
        for j in neighbors:
            register_subset((i, j), 'center_knn')

        local_pool = neighbors[:int(cfg['proposal_neighbor_pool'])]
        triple_count = 0
        for pos_a, j in enumerate(local_pool):
            for k in local_pool[pos_a + 1:]:
                register_subset((i, j, k), 'local_triple')
                triple_count += 1
                if triple_count >= int(cfg['proposal_max_triples_per_anchor']):
                    break
            if triple_count >= int(cfg['proposal_max_triples_per_anchor']):
                break

        quad_count = 0
        for pos_a, j in enumerate(local_pool):
            for pos_b, k in enumerate(local_pool[pos_a + 1:], start=pos_a + 1):
                for l in local_pool[pos_b + 1:]:
                    register_subset((i, j, k, l), 'local_quad')
                    quad_count += 1
                    if quad_count >= int(cfg['proposal_max_quads_per_anchor']):
                        break
                if quad_count >= int(cfg['proposal_max_quads_per_anchor']):
                    break
            if quad_count >= int(cfg['proposal_max_quads_per_anchor']):
                break

    candidate_df = pd.DataFrame(candidate_map.values()) if candidate_map else pd.DataFrame()
    if candidate_df.empty:
        return candidate_df, candidate_map

    candidate_df['proposal_source_count'] = candidate_df['proposal_sources'].map(len)
    candidate_df['proposal_sources'] = candidate_df['proposal_sources'].map(lambda values: ','.join(sorted(values)))
    candidate_df = candidate_df.sort_values(
        ['subset_size', 'member_count_before', 'proposal_source_count', 'center_dist', 'n_anchor_centers'],
        ascending=[False, False, False, True, True],
    ).reset_index(drop=True)
    return candidate_df, candidate_map


def run_iterative_capsule_merging(target_label: int, initial_regions, cfg):
    current_regions = [
        {
            **region,
            'center': np.asarray(region['center'], dtype=np.float32),
            'centers': np.asarray(region['centers'], dtype=np.float32),
            'member_count': int(region['member_count']),
            'source_ids': list(region['source_ids']),
            'source_eps_map': dict(region['source_eps_map']),
            'contained_source_ids': list(region['contained_source_ids']),
            'uncontained_source_ids': list(region['uncontained_source_ids']),
            'min_source_eps': float(region['min_source_eps']),
            'eps_cap_upper': float(region['eps_cap_upper']),
            'certified_eps': float(region['certified_eps']),
            'margin_lower': float(region['margin_lower']) if not np.isnan(region['margin_lower']) else np.nan,
        }
        for region in initial_regions
    ]
    pass_rows = []
    merge_rows = []

    for pass_idx in range(1, int(cfg['iter_max_passes']) + 1):
        candidate_df, candidate_map = build_local_capsule_candidates(current_regions, cfg)
        if candidate_df.empty:
            surviving_eps = np.asarray([float(region['certified_eps']) for region in current_regions], dtype=np.float64)
            pass_rows.append({
                'target_label': int(target_label),
                'pass_idx': pass_idx,
                'regions_start': int(len(current_regions)),
                'candidate_capsules': 0,
                'candidates_tested': 0,
                'successful_merges': 0,
                'regions_end': int(len(current_regions)),
                'accepted_eps_median': np.nan,
                'surviving_eps_mean': float(surviving_eps.mean()) if len(surviving_eps) else np.nan,
                'surviving_eps_median': float(np.median(surviving_eps)) if len(surviving_eps) else np.nan,
            })
            break

        shortlisted = pd.concat([
            candidate_df.loc[candidate_df['subset_size'] == 2].head(int(cfg['iter_top_pairs_per_pass'])),
            candidate_df.loc[candidate_df['subset_size'] == 3].head(int(cfg['iter_top_triples_per_pass'])),
            candidate_df.loc[candidate_df['subset_size'] == 4].head(int(cfg['iter_top_quads_per_pass'])),
        ], ignore_index=True)
        shortlisted = shortlisted.sort_values(
            ['subset_size', 'member_count_before', 'proposal_source_count', 'center_dist', 'n_anchor_centers'],
            ascending=[False, False, False, True, True],
        ).reset_index(drop=True)

        certified_candidates = []
        candidates_tested = 0
        for _, row in shortlisted.iterrows():
            candidate_obj = candidate_map[row['candidate_key']]
            candidates_tested += 1
            cert = binary_search_capsule_eps(
                int(target_label),
                candidate_obj['centers'],
                eps_hi=float(candidate_obj['eps_cap_upper']),
                steps=int(cfg['capsule_binary_search_steps']),
            )
            if not cert['certified'] or float(cert['eps']) <= float(cfg['capsule_eps_floor']):
                continue

            source_ids = candidate_obj['source_ids']
            source_eps_map = candidate_obj['source_eps_map']
            contained_source_ids = sorted([sid for sid in source_ids if float(source_eps_map[sid]) <= float(cert['eps'])])
            uncontained_source_ids = sorted([sid for sid in source_ids if sid not in contained_source_ids])
            n_sources_total = len(source_ids)
            n_sources_surely_contained = len(contained_source_ids)
            contained_fraction = (n_sources_surely_contained / n_sources_total) if n_sources_total > 0 else np.nan

            certified_candidates.append({
                **row.to_dict(),
                'cert': cert,
                'candidate_obj': candidate_obj,
                'n_sources_total': int(n_sources_total),
                'n_sources_surely_contained': int(n_sources_surely_contained),
                'contained_fraction': float(contained_fraction),
                'fully_contained': bool(n_sources_surely_contained == n_sources_total),
                'contained_source_ids': contained_source_ids,
                'uncontained_source_ids': uncontained_source_ids,
            })

        certified_candidates = sorted(
            certified_candidates,
            key=lambda row: (
                -int(row['member_count_before']),
                -int(row['subset_size']),
                -float(row['contained_fraction']),
                -float(row['cert']['eps']),
                -float(row['cert']['margin_lower']),
                float(row['center_dist']),
            ),
        )

        used = set()
        next_regions = []
        successful_merges = 0
        accepted_eps = []
        for row in certified_candidates:
            candidate_obj = row['candidate_obj']
            region_indices = tuple(int(idx) for idx in candidate_obj['region_indices'])
            if any(idx in used for idx in region_indices):
                continue
            merged_region = {
                'region_key': f'class{target_label}_pass{pass_idx}_merge{successful_merges + 1}',
                'center': candidate_obj['centers'].mean(axis=0),
                'centers': candidate_obj['centers'].astype(np.float32),
                'member_count': int(candidate_obj['member_count_before']),
                'source_ids': list(candidate_obj['source_ids']),
                'source_eps_map': dict(candidate_obj['source_eps_map']),
                'contained_source_ids': list(row['contained_source_ids']),
                'uncontained_source_ids': list(row['uncontained_source_ids']),
                'min_source_eps': float(candidate_obj['min_source_eps']),
                'eps_cap_upper': float(row['cert']['eps']),
                'certified_eps': float(row['cert']['eps']),
                'margin_lower': float(row['cert']['margin_lower']),
            }
            next_regions.append(merged_region)
            used.update(region_indices)
            successful_merges += 1
            accepted_eps.append(float(row['cert']['eps']))
            merge_rows.append({
                'target_label': int(target_label),
                'pass_idx': int(pass_idx),
                'candidate_key': row['candidate_key'],
                'subset_size': int(row['subset_size']),
                'member_count_before': int(row['member_count_before']),
                'proposal_source_count': int(row['proposal_source_count']),
                'proposal_sources': row['proposal_sources'],
                'center_dist': float(row['center_dist']),
                'n_anchor_centers': int(row['n_anchor_centers']),
                'certified_eps': float(row['cert']['eps']),
                'margin_lower': float(row['cert']['margin_lower']),
                'min_source_eps': float(candidate_obj['min_source_eps']),
                'n_sources_total': int(row['n_sources_total']),
                'n_sources_surely_contained': int(row['n_sources_surely_contained']),
                'contained_fraction': float(row['contained_fraction']),
                'fully_contained': bool(row['fully_contained']),
            })

        for idx, region in enumerate(current_regions):
            if idx not in used:
                next_regions.append(region)

        surviving_eps = np.asarray([float(region['certified_eps']) for region in next_regions], dtype=np.float64)
        pass_rows.append({
            'target_label': int(target_label),
            'pass_idx': int(pass_idx),
            'regions_start': int(len(current_regions)),
            'candidate_capsules': int(len(candidate_df)),
            'candidates_tested': int(candidates_tested),
            'successful_merges': int(successful_merges),
            'regions_end': int(len(next_regions)),
            'accepted_eps_median': float(np.median(accepted_eps)) if accepted_eps else np.nan,
            'surviving_eps_mean': float(surviving_eps.mean()) if len(surviving_eps) else np.nan,
            'surviving_eps_median': float(np.median(surviving_eps)) if len(surviving_eps) else np.nan,
        })

        current_regions = next_regions
        if successful_merges == 0:
            break

    return pd.DataFrame(pass_rows), pd.DataFrame(merge_rows), current_regions


CLASS_RESULTS = {}
PASS_TABLES = []
MERGE_TABLES = []
FINAL_REGION_ROWS = []
CLASS_SUMMARIES = []

for target_label in sorted(np.unique(Y_SUPPORT)):
    initial_regions = make_initial_capsule_regions(ATLAS, int(target_label))
    pass_df, merge_df, final_regions = run_iterative_capsule_merging(int(target_label), initial_regions, RUN_CFG)
    CLASS_RESULTS[int(target_label)] = {
        'initial_regions': initial_regions,
        'final_regions': final_regions,
        'pass_df': pass_df,
        'merge_df': merge_df,
    }
    PASS_TABLES.append(pass_df)
    if not merge_df.empty:
        MERGE_TABLES.append(merge_df)

    final_member_counts = np.asarray([int(region['member_count']) for region in final_regions], dtype=np.int64)
    final_eps = np.asarray([float(region['certified_eps']) for region in final_regions], dtype=np.float64)
    contained_counts = np.asarray([len(region['contained_source_ids']) for region in final_regions], dtype=np.int64)
    total_counts = np.asarray([len(region['source_ids']) for region in final_regions], dtype=np.int64)
    contained_fracs = contained_counts / np.maximum(total_counts, 1)

    for region in final_regions:
        FINAL_REGION_ROWS.append({
            'target_label': int(target_label),
            'region_key': region['region_key'],
            'member_count': int(region['member_count']),
            'n_anchor_centers': int(region['centers'].shape[0]),
            'certified_eps': float(region['certified_eps']),
            'margin_lower': float(region['margin_lower']) if not np.isnan(region['margin_lower']) else np.nan,
            'min_source_eps': float(region['min_source_eps']),
            'n_sources_total': int(len(region['source_ids'])),
            'n_sources_surely_contained': int(len(region['contained_source_ids'])),
            'contained_fraction': float(len(region['contained_source_ids']) / max(len(region['source_ids']), 1)),
            'fully_contained': bool(len(region['contained_source_ids']) == len(region['source_ids'])),
        })

    fully_contained_merge_fraction = float(merge_df['fully_contained'].mean()) if not merge_df.empty else np.nan
    CLASS_SUMMARIES.append({
        'target_label': int(target_label),
        'initial_region_count': int(len(initial_regions)),
        'final_region_count': int(len(final_regions)),
        'compression_ratio': float(len(initial_regions) / max(len(final_regions), 1)),
        'successful_merges_total': int(pass_df['successful_merges'].sum()) if not pass_df.empty else 0,
        'passes_executed': int(len(pass_df)),
        'max_member_count_final': int(final_member_counts.max()) if len(final_member_counts) else 0,
        'mean_member_count_final': float(final_member_counts.mean()) if len(final_member_counts) else np.nan,
        'merged_region_fraction_final': float((final_member_counts > 1).mean()) if len(final_member_counts) else np.nan,
        'mean_certified_eps_final': float(final_eps.mean()) if len(final_eps) else np.nan,
        'median_certified_eps_final': float(np.median(final_eps)) if len(final_eps) else np.nan,
        'p10_certified_eps_final': float(np.quantile(final_eps, 0.10)) if len(final_eps) else np.nan,
        'p90_certified_eps_final': float(np.quantile(final_eps, 0.90)) if len(final_eps) else np.nan,
        'max_certified_eps_final': float(final_eps.max()) if len(final_eps) else np.nan,
        'surely_contained_source_fraction_final': float(contained_counts.sum() / max(total_counts.sum(), 1)),
        'fully_contained_region_fraction_final': float(np.mean(contained_fracs == 1.0)) if len(contained_fracs) else np.nan,
        'fully_contained_merge_fraction': fully_contained_merge_fraction,
    })

PASS_DF = pd.concat(PASS_TABLES, ignore_index=True) if PASS_TABLES else pd.DataFrame()
MERGE_DF = pd.concat(MERGE_TABLES, ignore_index=True) if MERGE_TABLES else pd.DataFrame()
FINAL_REGION_DF = pd.DataFrame(FINAL_REGION_ROWS)
CLASS_SUMMARY_DF = pd.DataFrame(CLASS_SUMMARIES).sort_values('target_label').reset_index(drop=True)

COMPRESSION_FRONTIER_DF = (
    PASS_DF[['target_label', 'pass_idx', 'regions_end', 'accepted_eps_median', 'surviving_eps_mean', 'surviving_eps_median']].copy()
    if not PASS_DF.empty else pd.DataFrame()
)

display(CLASS_SUMMARY_DF)
if not PASS_DF.empty:
    display(PASS_DF)
if not MERGE_DF.empty:
    display(MERGE_DF.head(30))
if not FINAL_REGION_DF.empty:
    display(FINAL_REGION_DF.head(30))


In [ ]:
GLOBAL_SUMMARY_DF = pd.DataFrame([{
    'atlas_build_seconds': float(ATLAS_BUILD_SECONDS),
    'total_initial_regions': int(CLASS_SUMMARY_DF['initial_region_count'].sum()),
    'total_final_regions': int(CLASS_SUMMARY_DF['final_region_count'].sum()),
    'global_compression_ratio': float(CLASS_SUMMARY_DF['initial_region_count'].sum() / max(CLASS_SUMMARY_DF['final_region_count'].sum(), 1)),
    'total_successful_merges': int(CLASS_SUMMARY_DF['successful_merges_total'].sum()),
    'max_member_count_overall': int(CLASS_SUMMARY_DF['max_member_count_final'].max()),
    'mean_member_count_overall': float(CLASS_SUMMARY_DF['mean_member_count_final'].mean()),
    'mean_certified_eps_overall': float(CLASS_SUMMARY_DF['mean_certified_eps_final'].mean()),
    'median_certified_eps_overall': float(np.average(CLASS_SUMMARY_DF['median_certified_eps_final'], weights=CLASS_SUMMARY_DF['final_region_count'])),
    'mean_surely_contained_source_fraction_final': float(CLASS_SUMMARY_DF['surely_contained_source_fraction_final'].mean()),
    'mean_fully_contained_region_fraction_final': float(CLASS_SUMMARY_DF['fully_contained_region_fraction_final'].mean()),
    'mean_fully_contained_merge_fraction': float(CLASS_SUMMARY_DF['fully_contained_merge_fraction'].mean()),
}])

display(GLOBAL_SUMMARY_DF)
print()
print('Per-class summary:')
display(CLASS_SUMMARY_DF)

if not MERGE_DF.empty:
    print()
    print('Containment by subset size:')
    display(
        MERGE_DF.groupby(['target_label', 'subset_size'])[['n_sources_total', 'n_sources_surely_contained', 'contained_fraction', 'certified_eps', 'margin_lower']]
        .agg(['count', 'mean', 'median'])
    )

    print()
    print('Merge-level tradeoff summary:')
    display(
        MERGE_DF.groupby(['target_label', 'subset_size'])[['member_count_before', 'n_anchor_centers', 'certified_eps', 'margin_lower']]
        .agg(['count', 'mean', 'median'])
    )

if not FINAL_REGION_DF.empty:
    print()
    print('Final capsule summary:')
    display(
        FINAL_REGION_DF.groupby('target_label')[['member_count', 'n_anchor_centers', 'certified_eps', 'contained_fraction']]
        .agg(['mean', 'median', 'min', 'max'])
    )

if not COMPRESSION_FRONTIER_DF.empty:
    print()
    print('Compression frontier by pass:')
    display(COMPRESSION_FRONTIER_DF)

if not MERGE_DF.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    sns.scatterplot(
        data=MERGE_DF,
        x='member_count_before',
        y='certified_eps',
        hue='target_label',
        style='subset_size',
        alpha=0.8,
        ax=axes[0],
    )
    axes[0].set_title('Certified eps vs member_count_before')
    axes[0].set_xlabel('member_count_before')
    axes[0].set_ylabel('certified_eps')

    sns.scatterplot(
        data=MERGE_DF,
        x='n_anchor_centers',
        y='certified_eps',
        hue='target_label',
        style='subset_size',
        alpha=0.8,
        ax=axes[1],
    )
    axes[1].set_title('Certified eps vs n_anchor_centers')
    axes[1].set_xlabel('n_anchor_centers')
    axes[1].set_ylabel('certified_eps')
    for ax in axes:
        ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()

    print()
    print('Certified eps quantiles (accepted merges):')
    display(
        MERGE_DF.groupby('target_label')['certified_eps']
        .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
    )
else:
    print('No certified Minkowski capsule merges were found under the current Adult-space settings.')


## Questions to ask after running

- How much certified compression do Minkowski capsules provide on Adult once we require nonzero full-dimensional thickness?
- How quickly does certified `eps` shrink as `member_count_before` or `n_anchor_centers` grows?
- How much of the original anchor-centered source-radius model is **surely contained** inside the final merged capsules?
- Are we mostly getting capsule compression, or are we also getting meaningful conservative source-region replacement?

Interpretation note:
- The containment numbers in this notebook are a **conservative proxy**, not exact high-dimensional polytope containment.
- A source region is counted as surely contained only when the merged capsule radius is at least as large as that source anchor's original scalar local radius.
